# 06 - Live Demo

End-to-end demonstration of the AeroConform real-time anomaly
detection system: pipeline construction, live monitoring,
MCP server tools, and processing a synthetic airspace snapshot.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from aeroconform.config import AeroConformConfig

config = AeroConformConfig()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Target FIR: {config.target_fir}")
print(f"Bbox: {config.bbox}")

## Build the Full AeroConform Pipeline

The pipeline chains:
1. Trajectory foundation model (pre-trained transformer)
2. Optional graph attention network
3. Calibrated conformal detector

In [ ]:
from aeroconform.models.trajectory_model import TrajectoryTransformer
from aeroconform.models.graph_attention import AirspaceGATv2
from aeroconform.models.conformal import AdaptiveConformalDetector
from aeroconform.models.pipeline import AeroConformPipeline
from aeroconform.data.preprocessing import (
    delta_encode, compute_norm_stats, normalize, NormStats,
)

# Initialize models
foundation_model = TrajectoryTransformer.from_config(config).to(device)
foundation_model.eval()
graph_model = AirspaceGATv2.from_config(config).to(device)
graph_model.eval()

# Calibrate detector with synthetic clean data
rng = np.random.default_rng(42)


def generate_synthetic_trajectory(n_steps: int, rng: np.random.Generator) -> np.ndarray:
    """Generate a single realistic synthetic ADS-B trajectory."""
    lat = rng.uniform(config.bbox[0], config.bbox[1])
    lon = rng.uniform(config.bbox[2], config.bbox[3])
    alt = rng.uniform(15000, 40000)
    vel = rng.uniform(200, 500)
    hdg = rng.uniform(0, 360)
    vrate = 0.0
    traj = np.zeros((n_steps, 6))
    for t in range(n_steps):
        traj[t] = [lat, lon, alt, vel, hdg, vrate]
        lat += vel * np.cos(np.radians(hdg)) / 3600 / 60
        lon += vel * np.sin(np.radians(hdg)) / 3600 / 60 / np.cos(np.radians(lat))
        alt += vrate / 60
        vel = np.clip(vel + rng.normal(0, 1), 150, 600)
        hdg = (hdg + rng.normal(0, 0.3)) % 360
        vrate = rng.normal(0, 50)
    return traj


clean_trajs = [generate_synthetic_trajectory(200, rng) for _ in range(20)]
train_deltas = [delta_encode(t) for t in clean_trajs]
norm_stats = compute_norm_stats(train_deltas)

# Compute calibration scores
detector = AdaptiveConformalDetector.from_config(config)
cal_scores = []

for traj in clean_trajs:
    deltas = delta_encode(traj)
    normed = normalize(deltas, norm_stats)
    padded = np.zeros((config.seq_len, 6))
    padded[:min(len(normed), config.seq_len)] = normed[:config.seq_len]

    x = torch.from_numpy(padded).float().unsqueeze(0).to(device)
    with torch.no_grad():
        means, log_vars, log_weights, _ = foundation_model(x)

    for p in range(means.shape[1] - 1):
        ts = (p + 1) * config.patch_len
        te = ts + config.patch_len
        if te > len(deltas):
            break
        target = padded[ts:te].flatten()
        score = detector.compute_nonconformity_score(
            target, means[0, p].cpu().numpy(),
            log_vars[0, p].cpu().numpy(),
            log_weights[0, p].cpu().numpy(),
        )
        cal_scores.append(score)

detector.calibrate(np.array(cal_scores[:config.cal_window]))

# Build pipeline
pipeline = AeroConformPipeline(
    model=foundation_model,
    detector=detector,
    config=config,
    graph_model=graph_model,
    norm_stats=norm_stats,
    device=device,
)

print("Pipeline constructed successfully.")
print(f"  Foundation model: {sum(p.numel() for p in foundation_model.parameters()):,} params")
print(f"  Graph model: {sum(p.numel() for p in graph_model.parameters()):,} params")
print(f"  Calibration scores: {len(cal_scores)}")
print(f"  Detection threshold: {detector.get_threshold():.4f}")

## Process a Single Trajectory

Run the pipeline on a clean trajectory and an anomalous one.

In [ ]:
from aeroconform.data.synthetic_anomalies import AnomalyInjector

# Clean trajectory
clean_traj = generate_synthetic_trajectory(200, np.random.default_rng(99))
clean_result = pipeline.detect_anomalies(clean_traj, update_calibration=False)

print("Clean trajectory result:")
print(f"  Is anomalous: {clean_result['is_anomalous']}")
print(f"  Max score: {clean_result['max_score']:.4f}")
print(f"  Mean score: {clean_result['mean_score']:.4f}")
print(f"  Anomalous patches: {clean_result['anomalous_patches']}")

# Anomalous trajectory (GPS spoofing)
injector = AnomalyInjector(rng=np.random.default_rng(42))
spoofed_traj, labels = injector.inject_gps_spoofing(clean_traj, offset_nm=10.0)
spoofed_result = pipeline.detect_anomalies(spoofed_traj, update_calibration=False)

print("\nGPS-spoofed trajectory result:")
print(f"  Is anomalous: {spoofed_result['is_anomalous']}")
print(f"  Max score: {spoofed_result['max_score']:.4f}")
print(f"  Mean score: {spoofed_result['mean_score']:.4f}")
print(f"  Anomalous patches: {spoofed_result['anomalous_patches']}")

## LiveAirspaceMonitor

The `LiveAirspaceMonitor` polls the OpenSky API, maintains per-aircraft
trajectory buffers, and runs the pipeline continuously. Here we
demonstrate it with a synthetic snapshot.

In [ ]:
from aeroconform.inference.live_monitor import LiveAirspaceMonitor

monitor = LiveAirspaceMonitor(pipeline=pipeline, config=config)

print("LiveAirspaceMonitor initialized.")
print(f"  Buffer max length: {monitor.buffer_maxlen}")
print(f"  Poll interval: {config.opensky_poll_interval}s")
print(f"  Tracked aircraft: {monitor.get_status()['tracked_aircraft']}")

## Process Synthetic Airspace Snapshot

Create a synthetic snapshot with multiple aircraft states and
feed it through the monitor.

In [ ]:
# Build multiple snapshots to accumulate trajectory buffers
n_aircraft = 8
n_snapshots = 50  # Need enough to build up trajectory buffers

# Generate base positions and velocities for each aircraft
aircraft_states = {}
for i in range(n_aircraft):
    icao24 = f"ac{i:04d}"
    aircraft_states[icao24] = {
        "lat": rng.uniform(config.bbox[0], config.bbox[1]),
        "lon": rng.uniform(config.bbox[2], config.bbox[3]),
        "alt": rng.uniform(20000, 38000),
        "vel": rng.uniform(250, 450),
        "hdg": rng.uniform(0, 360),
        "vrate": 0.0,
    }

all_alerts = []
for snap_idx in range(n_snapshots):
    # Build snapshot DataFrame
    rows = []
    for icao24, state in aircraft_states.items():
        rows.append({
            "icao24": icao24,
            "latitude": state["lat"],
            "longitude": state["lon"],
            "baro_altitude": state["alt"],
            "velocity": state["vel"],
            "true_track": state["hdg"],
            "vertical_rate": state["vrate"],
            "on_ground": False,
            "callsign": icao24.upper(),
        })

    snapshot_df = pd.DataFrame(rows)
    alerts = monitor.process_snapshot(snapshot_df)
    all_alerts.extend(alerts)

    # Update aircraft positions (simulate movement)
    for icao24, state in aircraft_states.items():
        state["lat"] += state["vel"] * np.cos(np.radians(state["hdg"])) / 3600 / 60
        state["lon"] += (state["vel"] * np.sin(np.radians(state["hdg"])) /
                         3600 / 60 / np.cos(np.radians(state["lat"])))
        state["alt"] += state["vrate"] / 60
        state["vel"] = np.clip(state["vel"] + rng.normal(0, 1), 150, 600)
        state["hdg"] = (state["hdg"] + rng.normal(0, 0.3)) % 360
        state["vrate"] = rng.normal(0, 50)

# Check status
status = monitor.get_status()
print(f"Monitor status after {n_snapshots} snapshots:")
print(f"  Tracked aircraft: {status['tracked_aircraft']}")
print(f"  Buffer sizes: {status['buffer_sizes']}")
print(f"  Total alerts generated: {len(all_alerts)}")

if all_alerts:
    print(f"\nSample alert:")
    alert = all_alerts[0]
    for k, v in alert.items():
        print(f"  {k}: {v}")

## Visualize Monitored Aircraft

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Plot trajectory buffers
for icao24, buffer in monitor.trajectory_buffers.items():
    traj = np.array(list(buffer))
    ax.plot(traj[:, 1], traj[:, 0], "-", alpha=0.6, linewidth=1.5, label=icao24)
    ax.plot(traj[-1, 1], traj[-1, 0], "o", markersize=8)

# Draw bounding box
bbox = config.bbox
ax.plot(
    [bbox[2], bbox[3], bbox[3], bbox[2], bbox[2]],
    [bbox[0], bbox[0], bbox[1], bbox[1], bbox[0]],
    "r--", linewidth=1.5, label="LIMM FIR",
)

# Mark alerts
for alert in all_alerts:
    pos = alert["position"]
    ax.plot(pos["lon"], pos["lat"], "rx", markersize=15, markeredgewidth=2)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Live Airspace Monitor: {n_aircraft} aircraft, {len(all_alerts)} alerts")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## MCP Server Tools

The `AeroConformMCPServer` exposes three tools to Claude:
1. `get_airspace_status` - Current anomaly status for a region
2. `get_aircraft_trajectory` - Trajectory and scores for one aircraft
3. `get_airspace_graph` - Current interaction graph

In [ ]:
from aeroconform.inference.mcp_server import AeroConformMCPServer

mcp_server = AeroConformMCPServer(monitor=monitor, config=config)

# List available tools
tools = mcp_server.get_tools()
print(f"MCP Server tools ({len(tools)}):")
for tool in tools:
    print(f"\n  {tool['name']}:")
    print(f"    {tool['description'][:80]}...")
    params = tool["parameters"].get("properties", {})
    if params:
        print(f"    Parameters: {list(params.keys())}")

## Tool 1: Get Airspace Status

In [ ]:
import json

status = mcp_server.get_airspace_status()
print("get_airspace_status():")
print(json.dumps(status, indent=2))

## Tool 2: Get Aircraft Trajectory

In [ ]:
# Query a tracked aircraft
tracked_icao = list(monitor.trajectory_buffers.keys())[0]
traj_result = mcp_server.get_aircraft_trajectory(icao24=tracked_icao)
print(f"get_aircraft_trajectory('{tracked_icao}'):")
print(json.dumps(traj_result, indent=2))

# Query an unknown aircraft
unknown_result = mcp_server.get_aircraft_trajectory(icao24="unknown")
print(f"\nget_aircraft_trajectory('unknown'):")
print(json.dumps(unknown_result, indent=2))

## Tool 3: Get Airspace Graph

In [ ]:
graph_result = mcp_server.get_airspace_graph()
print("get_airspace_graph():")
print(f"  Timestamp: {graph_result['timestamp']}")
print(f"  Nodes: {graph_result['n_nodes']}")
print(f"  Edges: {graph_result['n_edges']}")
print(f"  Bbox: {graph_result['bbox']}")
if graph_result["nodes"]:
    print(f"\n  Sample node:")
    print(json.dumps(graph_result["nodes"][0], indent=4))

## End-to-End: Inject Anomaly and Detect

Demonstrate detection of a GPS spoofing attack injected into the
live monitoring stream.

In [ ]:
# Pick an aircraft to spoof
target_icao = list(monitor.trajectory_buffers.keys())[0]
target_buffer = list(monitor.trajectory_buffers[target_icao])
original_traj = np.array(target_buffer)

# Apply GPS spoofing to future states
injector = AnomalyInjector(rng=np.random.default_rng(123))
spoofed_traj, _ = injector.inject_gps_spoofing(original_traj, offset_nm=8.0)

# Feed spoofed states into the monitor
spoofed_alerts = []
for t in range(len(spoofed_traj)):
    state = spoofed_traj[t]
    snapshot = pd.DataFrame([{
        "icao24": target_icao,
        "latitude": state[0],
        "longitude": state[1],
        "baro_altitude": state[2],
        "velocity": state[3],
        "true_track": state[4],
        "vertical_rate": state[5],
        "on_ground": False,
        "callsign": target_icao.upper(),
    }])
    alerts = monitor.process_snapshot(snapshot)
    spoofed_alerts.extend(alerts)

print(f"GPS spoofing injection on {target_icao}:")
print(f"  Spoofed timesteps: {len(spoofed_traj)}")
print(f"  Alerts generated: {len(spoofed_alerts)}")

if spoofed_alerts:
    print(f"\n  First alert:")
    for k, v in spoofed_alerts[0].items():
        print(f"    {k}: {v}")

## Visualize Spoofed vs Clean Detection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Spatial comparison
ax = axes[0]
ax.plot(original_traj[:, 1], original_traj[:, 0], "b-", label="Clean", alpha=0.7)
ax.plot(spoofed_traj[:, 1], spoofed_traj[:, 0], "r-", label="Spoofed", alpha=0.7)
for alert in spoofed_alerts:
    pos = alert["position"]
    ax.plot(pos["lon"], pos["lat"], "rx", markersize=12, markeredgewidth=2)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"{target_icao}: Clean vs Spoofed")
ax.legend()
ax.grid(True, alpha=0.3)

# Score comparison
ax = axes[1]
clean_result = pipeline.detect_anomalies(original_traj, update_calibration=False)
spoofed_result = pipeline.detect_anomalies(spoofed_traj, update_calibration=False)

if clean_result["results"]:
    clean_scores = [r["score"] for r in clean_result["results"]]
    ax.plot(clean_scores, "b-", label="Clean scores", alpha=0.7)

if spoofed_result["results"]:
    spoof_scores = [r["score"] for r in spoofed_result["results"]]
    ax.plot(spoof_scores, "r-", label="Spoofed scores", alpha=0.7)

threshold = detector.get_threshold()
ax.axhline(threshold, color="orange", linestyle="--", label=f"Threshold ({threshold:.2f})")
ax.set_xlabel("Patch Index")
ax.set_ylabel("Non-conformity Score")
ax.set_title("Detection Scores")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Pipeline Summary Statistics

In [ ]:
print("AeroConform System Summary")
print("=" * 50)
print(f"\nFoundation Model:")
print(f"  Parameters: {sum(p.numel() for p in foundation_model.parameters()):,}")
print(f"  d_model: {config.d_model}")
print(f"  Layers: {config.n_layers}")
print(f"  Heads: {config.n_heads}")
print(f"  GMM components: {config.n_components}")

print(f"\nGraph Attention Network:")
print(f"  Parameters: {sum(p.numel() for p in graph_model.parameters()):,}")
print(f"  Layers: {config.graph_layers}")
print(f"  Heads: {config.graph_heads}")
print(f"  Edge features: {config.edge_dim}")

print(f"\nConformal Detector:")
print(f"  Alpha (target FAR): {config.alpha}")
print(f"  Calibration window: {config.cal_window}")
print(f"  Current threshold: {detector.get_threshold():.4f}")

print(f"\nMonitoring:")
print(f"  Tracked aircraft: {monitor.get_status()['tracked_aircraft']}")
print(f"  MCP tools: {len(mcp_server.get_tools())}")
print(f"  Device: {device}")

## Summary

This notebook demonstrated the full AeroConform system:

1. **Pipeline construction**: foundation model + graph + conformal detector
2. **Single trajectory processing**: clean vs anomalous detection
3. **LiveAirspaceMonitor**: processing streaming airspace snapshots
4. **MCP server tools**: 3 tools for Claude integration
   - `get_airspace_status`: regional anomaly overview
   - `get_aircraft_trajectory`: per-aircraft analysis
   - `get_airspace_graph`: interaction graph
5. **End-to-end spoofing detection**: inject + detect + visualize

The system provides real-time anomaly detection with statistical
false alarm rate guarantees via conformal prediction.